# McLaren P35 — collect / init / validate / I-O (workstation `cXBNn`)

Tag: `crashPostProcOK` (18 OFAT extracts, AI extractor 0.1.14).

This notebook is the recorded recipe. The unattended copy is
`scripts/ws_p35_probes/collect_init_validate.py`.

**Overlay:** `/enc/udeprod_cXBNn/work/rescale-ai` (`feat/transient-global-ts-outputs` + `file_utils` clock-key skip).

Do **not** open the Variables UI on this box (OOM history). Patch metadata via the last cell / `patch_variables.py`.


In [ ]:
import os, sys
from pathlib import Path

OVERLAY = Path("/enc/udeprod_cXBNn/work/rescale-ai")
SCRIPTS = Path("/enc/udeprod_cXBNn/work/mclaren_p35_probes/scripts")
os.environ.setdefault("PYTHONPATH", str(OVERLAY))
sys.path.insert(0, str(OVERLAY))
sys.path.insert(0, str(SCRIPTS))

print("PYTHONPATH overlay:", OVERLAY.exists())
print("scripts:", list(SCRIPTS.glob("*.py")))


## 1. Search jobs by tag

In [ ]:
from rescale_ai.interactive import DatasetBuilder
from config import DATASET_NAME, TAG, EXPECTED_JOB_IDS, FILE_PATTERNS, NUM_WORKERS_COLLECT, NUM_WORKERS_INIT

builder = DatasetBuilder()
print("HPS", builder.hps.link_path)

jobs = builder.search_jobs(tag=TAG, limit=100)

def jid(job):
    return getattr(job, "job_id", None) or job._initial_data.get("id", "")

found = {jid(j): j for j in jobs}
print(f"tagged={len(found)} expected={len(EXPECTED_JOB_IDS)}")
missing = [i for i in EXPECTED_JOB_IDS if i not in found]
extra = sorted(set(found) - set(EXPECTED_JOB_IDS))
print("missing", missing)
print("extra", extra)
if missing:
    for j in builder.job_explorer.get_by_ids(missing):
        found[jid(j)] = j
ordered = [found[i] for i in EXPECTED_JOB_IDS]
assert len(ordered) == 18, len(ordered)
builder.last_jobs = ordered
builder.show_jobs_info(ordered)


## 2. `collect_cases` (num_workers=1 — VTPs ~700 MB each)

In [ ]:
scratch = builder.collect_cases(
    name=DATASET_NAME,
    jobs=ordered,
    file_patterns=FILE_PATTERNS,
    num_workers=NUM_WORKERS_COLLECT,
    verbose=True,
)
print("scratch", scratch)


## 3. `initialize_dataset` (fea-deform)

Uses overlay `file_utils` so `timesteps` / `times` / `frame_indices` are **not** expanded into a junk Variables group.


In [ ]:
builder.initialize_dataset(
    name=DATASET_NAME,
    dataset_type="fea-deform",
    surface_file_name="case_data.vtp",
    num_workers=NUM_WORKERS_INIT,
)


## 4. `validate_dataset`

In [ ]:
final = builder.validate_dataset(
    DATASET_NAME,
    num_workers=NUM_WORKERS_INIT,
    upload_to_platform=True,
    description="McLaren P35 ES2 side-pole, 18 OFAT extracts (crashPostProcOK), displacement + occupant probe globals",
)
print("dataset", final)


## 5. Enable knobs + selected outputs

**Inputs (geometry knobs):** door inner/outer, TTF, airbag E22 / z-up, bracket cut, woodfibre trim, brace P16, cavity foam, rail foam, seat inbound foam.

**Node output:** displacement only.

**Global TS outputs (28):** head acc xyz, neck upper F/M xyz, T1/T12 acc xyz, rib defl U/M/L, rib VC U/M/L, shoulder force xyz, pelvis acc xyz, pubic Fy.

Disabled on purpose: `psa_description`, `seat_clubsport`, backplate, T12 loadcell, pubic Fx/Fz (not requested).


In [ ]:
from patch_variables import find_meta, patch
import yaml, json

meta_path = find_meta(Path(final))
meta = yaml.safe_load(meta_path.read_text())
report = patch(meta)
meta_path.write_text(yaml.safe_dump(meta, sort_keys=False, default_flow_style=False))
print("patched", meta_path)
print(json.dumps({k: (len(v) if isinstance(v, list) else v) for k, v in report.items()}, indent=2))
print("missing knobs", report["missing_knobs"])
print("missing globals", report["missing_globals"])


## 6. Sanity: 18 cases, T, knobs, probes

Holdouts from last time (do **not** hold unique-knob-only cases):

- `KwXUX` (parent `gfkFPb`) — door inner 1.3 mm
- `dWNQX` (parent `OUjFPb`) — cavity foam 60 gl


In [ ]:
import yaml
from config import HOLDOUT_JOB_IDS, KNOB_INPUTS, GLOBAL_OUTPUT_BASES

cases = sorted(p for p in (Path(final) / "cases").iterdir() if p.is_dir() and p.name.startswith("case"))
print("n_cases", len(cases))
rows = []
for c in cases:
    yml = yaml.safe_load((c / "case_data.yml").read_text()) or {}
    ts = yml.get("timeseries") or {}
    T = len(ts.get("timesteps") or ts.get("times") or [])
    n_ch = len([k for k in ts if k not in {"timesteps", "times", "frame_indices", "vtp_array_names"}])
    knobs = {k: yml.get(k) for k in KNOB_INPUTS}
    hold = any(h in c.name for h in HOLDOUT_JOB_IDS)
    rows.append((c.name, T, n_ch, hold, knobs))
    print(c.name, "T", T, "channels", n_ch, "HOLDOUT" if hold else "train")

# unique-knob freeze check: each train knob must still have >1 unique value
from collections import defaultdict
vals = defaultdict(set)
for name, T, n_ch, hold, knobs in rows:
    if hold:
        continue
    for k, v in knobs.items():
        vals[k].add(v)
print("--- train unique counts (must be >1 for every swept knob) ---")
for k in KNOB_INPUTS:
    print(k, len(vals[k]), sorted(vals[k], key=lambda x: str(x)))
